# Phase 1 -- p1_05: Model A test predictions for Codabench

Loads Model A's trained checkpoint (step 798), tries a few output fixes on
the same 100 dev examples A scored **0.8237** on, keeps the best, and
generates the official test predictions -> `submission.zip`.

**Setup:** GPU T4 x2 (only one is used). Internet on.
Inputs: `legallm-phase1-data`, `legallm-tokenizers`, `legallm-checkpoints`
(**latest version** -- click the orange refresh icon if shown).
Secrets on: `HF` (Hugging Face token, to download the test file).

**Time:** roughly 20-30 minutes. No training.


## 1. Imports and inputs

In [1]:
import json
import math
import shutil
import subprocess
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE)

KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/phase1_predict_a")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_STEP = 798        # A's finished run: A_step798_tokens52297728.pt
TRAIN_SEQ_LEN = 1024       # A was trained on 1024-token windows


def find_input_dir(marker, dataset_label):
    hits = sorted(KAGGLE_INPUT.rglob(marker)) if KAGGLE_INPUT.exists() else []
    if not hits:
        attached = sorted(p.name for p in KAGGLE_INPUT.glob("*")) if KAGGLE_INPUT.exists() else []
        raise FileNotFoundError(f"Could not find '{marker}'. Attach '{dataset_label}'. Attached now: {attached}")
    return hits[0].parent


def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def find_latest_checkpoint(prefix):
    """Newest '{prefix}_step*.pt' under /kaggle/input (any depth), or inside a .zip."""
    pts = list(KAGGLE_INPUT.rglob(f"{prefix}_step*.pt")) if KAGGLE_INPUT.exists() else []
    if pts:
        return max(pts, key=step_of)
    best = None
    for zpath in (KAGGLE_INPUT.rglob("*.zip") if KAGGLE_INPUT.exists() else []):
        try:
            with zipfile.ZipFile(zpath) as zf:
                for name in zf.namelist():
                    base = Path(name).name
                    if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                        if best is None or step_of(base) > best[0]:
                            best = (step_of(base), zpath, name)
        except zipfile.BadZipFile:
            continue
    if best is None:
        return None
    _, zpath, name = best
    dest = OUTPUT_DIR / Path(name).name
    print(f"Extracting {Path(name).name} from {zpath.name} ...")
    with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
        shutil.copyfileobj(src, dst)
    return dest


INPUT_DATA_DIR = find_input_dir("dev_prompts.json", "legallm-phase1-data")
INPUT_TOKENIZER_DIR = find_input_dir("t1_bpe.json", "legallm-tokenizers")
CKPT_PATH = find_latest_checkpoint("A")
assert CKPT_PATH is not None, "No A_step*.pt found -- attach legallm-checkpoints (latest version)."
if step_of(CKPT_PATH) != EXPECTED_STEP:
    raise RuntimeError(
        f"Found {Path(CKPT_PATH).name}, expected step {EXPECTED_STEP}. The attached legallm-checkpoints is "
        f"probably an older version: click the refresh icon next to it in the Input panel (or re-add it) "
        f"and re-run. (Set EXPECTED_STEP = {step_of(CKPT_PATH)} only if you really mean to use that one.)"
    )
print("Checkpoint:", CKPT_PATH)


PyTorch 2.10.0+cu128 | device: cuda:0
Checkpoint: /kaggle/input/datasets/jatiniitk/legallm-checkpoints/checkpoints/A_step798_tokens52297728.pt


## 2. Tokenizer and the official test prompts

The test file is downloaded from the official Hugging Face dataset with
your `HF` token (the dataset is gated).


In [2]:
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

tokenizer = Tokenizer.from_file(str(INPUT_TOKENIZER_DIR / "t1_bpe.json"))
EOS_ID = tokenizer.token_to_id("<eos>")
assert EOS_ID is not None

hf_token = UserSecretsClient().get_secret("HF")
test_path = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_1_Pretraining/Test/test.json",
                            repo_type="dataset", token=hf_token)
with open(test_path) as f:
    test_prompts = json.load(f)
assert isinstance(test_prompts, dict) and all(isinstance(v, dict) and "text" in v for v in test_prompts.values())
test_ids = list(test_prompts)
print(f"Test prompts: {len(test_ids)}  (first IDs: {test_ids[:3]})")

with open(INPUT_DATA_DIR / "dev_prompts.json") as f:
    dev_prompts = json.load(f)
with open(INPUT_DATA_DIR / "dev_truth.json") as f:
    dev_truth = json.load(f)
dev_ids = sorted(set(dev_prompts) & set(dev_truth))[:100]   # same 100 IDs A was scored on
print(f"Dev examples for comparison: {len(dev_ids)}")


test.json: 0.00B [00:00, ?B/s]

Test prompts: 1000  (first IDs: ['1', '2', '3'])
Dev examples for comparison: 100


## 3. Model A (same architecture as p1_03) and its weights

Same classes and parameter names as the training notebook, so the saved
weights load exactly. For generation the attention also accepts a
key-padding mask and per-row positions, which lets several prompts of
different lengths be generated together (left-padded).


In [3]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: [B, H, T, d]; cos/sin: [B, T, d/2] (per-row positions)."""
    x1, x2 = x[..., 0::2].float(), x[..., 1::2].float()      # rotate in fp32, as in training
    cos, sin = cos[:, None, :, :].float(), sin[:, None, :, :].float()
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).to(x.dtype)


class GQAAttention(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, d_head):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)

    def forward(self, x, cos, sin, cache, attn_mask):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k, v = cache.update(k, v)
        n_rep = self.n_head // self.n_kv_head
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, d_head, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head, d_head)
        self.mlp_norm = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)

    def forward(self, x, cos, sin, cache, attn_mask):
        x = x + self.attn(self.attn_norm(x), cos, sin, cache, attn_mask)
        return x + self.mlp(self.mlp_norm(x))


class KVCache:
    def __init__(self, batch_size, n_kv_head, max_len, d_head, device, dtype):
        self.k = torch.zeros(batch_size, n_kv_head, max_len, d_head, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config["d_model"], config["n_head"], config["n_kv_head"], config["d_head"], config["d_ff"])
            for _ in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def step(self, ids, positions, caches, attn_mask):
        """ids/positions: [B, T]. attn_mask: bool [B, 1, T, cache_len + T]."""
        x = self.embed(ids)
        cos, sin = self.rope_cos[positions], self.rope_sin[positions]
        for blk, cache in zip(self.blocks, caches):
            x = blk(x, cos, sin, cache, attn_mask)
        return self.head(self.norm_f(x))


ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
MODEL_CONFIG = ckpt["config"]
print("Config:", MODEL_CONFIG, "| step", ckpt["step"], "| tokens", f"{ckpt['tokens_seen']:,}")
model = MiniGPT(MODEL_CONFIG)
model.load_state_dict(ckpt["model_state"])
del ckpt
model = model.to(DEVICE).eval()
print(f"Loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")


Config: {'vocab_size': 32000, 'd_model': 896, 'n_layer': 16, 'n_head': 14, 'n_kv_head': 2, 'd_head': 64, 'd_ff': 2432, 'max_seq_len': 2048} | step 798 | tokens 52,297,728
Loaded: 162.7M params


## 4. Generation with output fixes

For each batch of prompts:
- **Context:** the prompt's last `TRAIN_SEQ_LEN - MAX_NEW_TOKENS` tokens, so
  prompt + continuation stays within the 1,024 positions A was trained on.
- **Length:** `<eos>` is blocked and generation continues until every
  prompt in the batch has more than 128 words (at most `MAX_NEW_TOKENS=320`
  tokens), then each is cut to exactly 128 words -- the reference length.
  (T1 splits every digit into its own token, so number-heavy text needs
  more tokens per word.)
- **Repetition control (optional):** a repetition penalty on tokens already
  *generated* (not the prompt -- copying names/sections from the prompt is
  good), and blocking any repeated n-gram inside the continuation.


In [4]:
MAX_NEW_TOKENS = 320   # cap; generation stops early once every row has > TARGET_WORDS words
TARGET_WORDS = 128


@torch.no_grad()
def generate_batch(prompt_id_lists, rep_penalty=1.0, no_repeat_ngram=0):
    """Greedy continuation for a list of token-id prompts. Returns token lists."""
    B = len(prompt_id_lists)
    max_prompt = TRAIN_SEQ_LEN - MAX_NEW_TOKENS
    prompts = [p[-max_prompt:] for p in prompt_id_lists]
    T0 = max(len(p) for p in prompts)
    pad = torch.tensor([T0 - len(p) for p in prompts], device=DEVICE)
    ids = torch.zeros(B, T0, dtype=torch.long, device=DEVICE)
    for b, p in enumerate(prompts):
        ids[b, T0 - len(p):] = torch.tensor(p, device=DEVICE)

    dtype = next(model.parameters()).dtype
    total = T0 + MAX_NEW_TOKENS
    caches = [KVCache(B, MODEL_CONFIG["n_kv_head"], total, MODEL_CONFIG["d_head"], DEVICE, dtype)
              for _ in range(MODEL_CONFIG["n_layer"])]
    idx = torch.arange(total, device=DEVICE)
    key_ok = idx[None, :] >= pad[:, None]                                 # [B, total]: False on padding
    positions = (idx[None, :] - pad[:, None]).clamp(min=0)               # [B, total]

    # Prefill: causal + no padding keys; each query may always see itself (no all-masked rows).
    causal = idx[:T0, None] >= idx[None, :T0]
    mask = (causal[None] & key_ok[:, None, :T0]) | torch.eye(T0, dtype=torch.bool, device=DEVICE)[None]
    logits = model.step(ids, positions[:, :T0], caches, mask[:, None])[:, -1, :].float()

    generated = [[] for _ in range(B)]
    seen_ngrams = [dict() for _ in range(B)]
    for t in range(MAX_NEW_TOKENS):
        logits[:, EOS_ID] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=DEVICE)
            scores = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(scores > 0, scores / rep_penalty, scores * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                banned = seen_ngrams[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if banned:
                    logits[b, list(banned)] = -float("inf")
        next_ids = logits.argmax(dim=-1)
        for b, tok in enumerate(next_ids.tolist()):
            generated[b].append(tok)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                gram = generated[b][-no_repeat_ngram:]
                seen_ngrams[b].setdefault(tuple(gram[:-1]), set()).add(gram[-1])
        if t == MAX_NEW_TOKENS - 1:
            break
        if t >= 160 and t % 16 == 0 and all(
                len(tokenizer.decode(g).split()) > TARGET_WORDS for g in generated):
            break
        cur = T0 + t
        step_mask = key_ok[:, None, None, :cur + 1]
        logits = model.step(next_ids[:, None], positions[:, cur:cur + 1], caches, step_mask)[:, -1, :].float()
    return generated


def to_words(token_ids):
    words = tokenizer.decode(token_ids).split()[:TARGET_WORDS]
    return " ".join(words) if words else "."


def predict(prompt_dict, ids, batch_size, desc, **settings):
    """Returns {id: {"text": ...}}. Prompts are grouped by length to limit padding."""
    encoded = {i: tokenizer.encode(prompt_dict[i]["text"]).ids for i in ids}
    order = sorted(ids, key=lambda i: len(encoded[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_size), desc=desc, dynamic_ncols=True):
        chunk = order[s:s + batch_size]
        for i, toks in zip(chunk, generate_batch([encoded[i] for i in chunk], **settings)):
            out[i] = {"text": to_words(toks)}
    return {i: out[i] for i in ids}


print("generate_batch / predict defined.")


generate_batch / predict defined.


## 5. Speed + correctness check for batched generation

Batched generation (16 prompts at a time) is ~10x faster than one at a
time. Check it first: the same 4 dev prompts generated together must give
the same tokens as generated one by one (in fp32). If they don't, the
notebook uses batch size 1 (slower, still correct). Then the model is
switched to fp16 for speed, and its agreement with fp32 is reported.


In [5]:
check_ids = dev_ids[:4]
check_prompts = [tokenizer.encode(dev_prompts[i]["text"]).ids for i in check_ids]
check_prompts[1] = check_prompts[1][:600]      # force different lengths -> real padding in the batch

single = [generate_batch([p])[0][:64] for p in check_prompts]
batched = [g[:64] for g in generate_batch(check_prompts)]
agree = sum(a == b for s, bt in zip(single, batched) for a, b in zip(s, bt)) / (64 * len(single))
print(f"fp32 batched vs one-at-a-time token agreement: {agree:.1%}")
BATCH_SIZE = 16 if agree >= 0.95 else 1
if BATCH_SIZE == 1:
    print("-> Batched generation disagreed; using batch size 1 (slower but safe).")

model = model.half()
model.rope_cos, model.rope_sin = model.rope_cos.float(), model.rope_sin.float()   # keep positions exact
fp16 = [g[:64] for g in generate_batch(check_prompts)]
agree16 = sum(a == b for s, h in zip(single, fp16) for a, b in zip(s, h)) / (64 * len(single))
print(f"fp16 vs fp32 token agreement: {agree16:.1%}")
if agree16 < 0.5:
    model = model.float()
    print("-> fp16 differs a lot from fp32; staying in fp32.")
print(f"Using batch size {BATCH_SIZE}, dtype {next(model.parameters()).dtype}.")


fp32 batched vs one-at-a-time token agreement: 100.0%
fp16 vs fp32 token agreement: 100.0%
Using batch size 16, dtype torch.float16.


## 6. Scoring (same metrics as evaluate_phase1.py)

In [6]:
try:
    import bert_score
except ImportError:
    subprocess.run(["pip", "install", "-q", "bert-score"], check=True)
    import bert_score


def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")


def rouge_l_f1(pred_text, ref_text):
    pt, rt = pred_text.split(), ref_text.split()
    if not pt or not rt:
        return 0.0
    l = lcs_length(pt, rt)
    if not l:
        return 0.0
    p, r = l / len(pt), l / len(rt)
    return 2 * p * r / (p + r)


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    clipped, totals = [0] * 4, [0] * 4
    pred_len = ref_len = 0
    for p, r in zip(preds, refs):
        pred_len += len(p)
        ref_len += len(r)
        for n in range(1, 5):
            pn, rn = ngrams(p, n), ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pred_len == 0:
        return 0.0
    bp = 1.0 if pred_len > ref_len else math.exp(1.0 - ref_len / pred_len)
    return bp * math.exp(sum(math.log(x) for x in prec) / 4)


def score(pred, truth):
    ids = list(truth)
    assert set(ids) == set(pred), "ID mismatch"
    p_txt = [pred[i]["text"].strip() for i in ids]
    r_txt = [truth[i]["text"].strip() for i in ids]
    _, _, f1 = bert_score.score(p_txt, r_txt, lang="en", batch_size=8, verbose=False,
                                device="cuda" if torch.cuda.is_available() else "cpu")
    return {
        "bertscore": float(f1.mean()),
        "rouge_l": sum(rouge_l_f1(p, r) for p, r in zip(p_txt, r_txt)) / len(ids),
        "bleu4": bleu4([t.split() for t in p_txt], [t.split() for t in r_txt]),
    }


print("Scoring functions ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00
Scoring functions ready.


## 7. Compare output-fix settings on the 100 dev examples

Reference: A's original generation (greedy, 128 tokens, no repetition
control) scored **BERTScore 0.8237, ROUGE-L 0.1548, BLEU-4 0.0283** on
these same 100 examples. The best setting below (by BERTScore, the
primary metric) is used for the test set.


In [7]:
SETTINGS = {
    "length_only":        dict(rep_penalty=1.0, no_repeat_ngram=0),
    "rep1.1_norepeat4":   dict(rep_penalty=1.1, no_repeat_ngram=4),
    "rep1.2_norepeat3":   dict(rep_penalty=1.2, no_repeat_ngram=3),
}
dev_truth_subset = {i: dev_truth[i] for i in dev_ids}
results = {}
for name, settings in SETTINGS.items():
    t0 = time.time()
    preds = predict(dev_prompts, dev_ids, BATCH_SIZE, f"dev:{name}", **settings)
    results[name] = score(preds, dev_truth_subset)
    results[name]["minutes"] = (time.time() - t0) / 60
    print(f"{name:20s} BERTScore={results[name]['bertscore']:.4f}  ROUGE-L={results[name]['rouge_l']:.4f}  "
          f"BLEU-4={results[name]['bleu4']:.4f}  ({results[name]['minutes']:.1f} min)")
    print("   sample:", preds[dev_ids[0]]["text"][:200], "...")

print(f"{'original (p1_03)':20s} BERTScore=0.8237  ROUGE-L=0.1548  BLEU-4=0.0283")
BEST = max(results, key=lambda k: (results[k]["bertscore"], results[k]["rouge_l"]))
print(f"\nBest setting: {BEST} -> used for the test set.")
with open(OUTPUT_DIR / "dev_settings_comparison.json", "w") as f:
    json.dump(results, f, indent=2)


dev:length_only:   0%|          | 0/7 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)

dev:rep1.2_norepeat3:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rep1.2_norepeat3     BERTScore=0.8245  ROUGE-L=0.1984  BLEU-4=0.0787  (1.0 min)
   sample: land; (2) the property of any person, who is a minor or a member of the State Government, in the case of a private property, shall be deemed to have been so transferred and all other rights which are  ...
original (p1_03)     BERTScore=0.8237  ROUGE-L=0.1548  BLEU-4=0.0283

Best setting: rep1.2_norepeat3 -> used for the test set.


## 8. Test predictions -> `submission.zip`

Checks before zipping: exactly the official test IDs, every text
non-empty, and `prediction.json` at the root of the zip.


In [8]:
t0 = time.time()
test_preds = predict(test_prompts, test_ids, BATCH_SIZE, "test", **SETTINGS[BEST])
print(f"Generated {len(test_preds)} test predictions in {(time.time() - t0) / 60:.1f} min.")

assert set(test_preds) == set(test_ids), "Prediction IDs must equal the official test IDs exactly."
assert all(isinstance(v["text"], str) and v["text"].strip() for v in test_preds.values()), "Empty text found."
word_counts = [len(v["text"].split()) for v in test_preds.values()]
print(f"Words per prediction: min {min(word_counts)}, mean {sum(word_counts) / len(word_counts):.1f}, max {max(word_counts)}")
print("Example:", json.dumps({test_ids[0]: test_preds[test_ids[0]]}, indent=1)[:400])

pred_path = OUTPUT_DIR / "prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(test_preds, f, ensure_ascii=False)
zip_path = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")
with zipfile.ZipFile(zip_path) as zf:
    assert zf.namelist() == ["prediction.json"], zf.namelist()
    assert set(json.loads(zf.read("prediction.json"))) == set(test_ids)
print(f"\nsubmission.zip ready ({zip_path.stat().st_size / 1e3:.0f} KB): contains only prediction.json at the root.")
print(f"Setting used: {BEST}  |  dev BERTScore {results[BEST]['bertscore']:.4f} (original: 0.8237)")


test:   0%|          | 0/63 [00:00<?, ?it/s]

Generated 1000 test predictions in 7.3 min.
Words per prediction: min 34, mean 127.1, max 128
Example: {
 "1": {
  "text": "not open to the High Court to take recourse of the power under Section 126(3). The question is whether the High Courts have jurisdiction over the suit and that too without any justification. In other words, the High court has rightly held that the land in question was acquired by the State Government for the purpose of acquisition of the land on the ground that it had been gra

submission.zip ready (159 KB): contains only prediction.json at the root.
Setting used: rep1.2_norepeat3  |  dev BERTScore 0.8245 (original: 0.8237)


In [9]:
from IPython.display import FileLink, display

print("Download (works in an interactive session; after Save & Run All use the Output tab):")
display(FileLink(str(zip_path.relative_to(Path.cwd())) if zip_path.is_relative_to(Path.cwd()) else str(zip_path)))


Download (works in an interactive session; after Save & Run All use the Output tab):


/kaggle/working/phase1_predict_a/submission.zip

## 9. Submit

1. Download `submission.zip` (link above, or the notebook's **Output** tab ->
   `phase1_predict_a/submission.zip`).
2. Codabench -> the competition -> **My Submissions** -> choose the Phase 1
   task -> upload `submission.zip`.
3. Scoring takes ~15-30 min. Note the score next to the dev BERTScore
   printed above.
